# Parte 01 · Carga y limpieza de datos — Dataset: Suicidios

**Proyecto final · Minería de Datos 1 · Grupo #2**
Colegio Universitario de Cartago — Tecnologías de la Información

**Dataset:** `global_suicide_rates_real_who_worldbank.csv`, guardado como `data/raw/suicidio.csv`. Contiene tasas de suicidio por país (datos de la OMS) junto con datos económicos de cada país (datos del Banco Mundial).

**¿Qué hacemos en esta parte?**
Cargamos los datos y los revisamos para ver qué tan "limpios" vienen. Buscamos:
- valores vacíos,
- filas repetidas,
- datos que no tienen sentido.

Después corregimos lo necesario y guardamos el resultado en `data/processed/suicidio_limpio.csv`. Ese es el archivo que usan las partes 02 a 05.

**Contenido**
1. Importación de librerías
2. Carga de datos
3. Dimensiones y vista general
4. Tipos de variables
5. ¿Qué representa cada fila?
6. Valores nulos
7. Registros duplicados
8. Inconsistencias
9. Valores atípicos
10. Limpieza de datos
11. Revisión final y exportación
12. Conclusiones

## 1. Importación de librerías

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Ocultamos advertencias y configuramos cómo se muestran las tablas
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)       # mostrar todas las columnas
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)  # números con 2 decimales

print("pandas:", pd.__version__, "| numpy:", np.__version__)

pandas: 2.3.3 | numpy: 2.4.3


## 2. Carga de datos

Usamos rutas relativas (con `../../`) para que el notebook funcione en la computadora de cualquier integrante del grupo. Leemos el archivo en UTF-8 para que los nombres con tildes salgan bien (por ejemplo, *Côte d'Ivoire* o *Türkiye*).

In [2]:
RUTA_RAW = Path("../../data/raw/suicidio.csv")
RUTA_PROCESSED = Path("../../data/processed/suicidio_limpio.csv")

df_raw = pd.read_csv(RUTA_RAW, encoding="utf-8")
df = df_raw.copy()   # trabajamos con una copia para no tocar los datos originales
df.head()

,country,country_code,year,sex,age_bracket,suicide_rate_per_100k,generation,gdp_usd,gdp_per_capita_usd,total_country_population
0,Afghanistan,AFG,2000,both,all_ages,4.36,All Cohorts,"3,521,418,059.92",174.93,"20,130,327.00"
1,Afghanistan,AFG,2000,female,all_ages,2.91,All Cohorts,"3,521,418,059.92",174.93,"20,130,327.00"
2,Afghanistan,AFG,2000,male,all_ages,5.79,All Cohorts,"3,521,418,059.92",174.93,"20,130,327.00"
3,Afghanistan,AFG,2001,both,all_ages,4.38,All Cohorts,"2,813,571,753.87",138.71,"20,284,307.00"
4,Afghanistan,AFG,2001,female,all_ages,2.92,All Cohorts,"2,813,571,753.87",138.71,"20,284,307.00"


## 3. Dimensiones y vista general

In [3]:
# shape nos dice cuántas filas y columnas hay
filas, columnas = df.shape
print(f"Registros (filas): {filas:,}")
print(f"Variables (columnas): {columnas}")
print(f"Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Registros (filas): 18,315
Variables (columnas): 10
Memoria utilizada: 6.30 MB


In [4]:
df.sample(5, random_state=42)  # 5 filas al azar

,country,country_code,year,sex,age_bracket,suicide_rate_per_100k,generation,gdp_usd,gdp_per_capita_usd,total_country_population
8241,Jordan,JOR,2008,both,all_ages,1.16,All Cohorts,"25,651,620,830.61","3,669.83","6,989,866.00"
6932,Haiti,HTI,2000,male,all_ages,9.50,All Cohorts,"6,813,566,098.87",820.60,"8,303,151.00"
9843,Madagascar,MDG,2014,both,all_ages,5.92,All Cohorts,"12,522,957,399.23",505.68,"24,764,348.00"
14968,Solomon Islands,SLB,2006,female,all_ages,8.53,All Cohorts,"617,257,457.76","1,255.62","491,594.00"
15134,Somalia,SOM,2021,female,all_ages,6.09,All Cohorts,"9,483,000,000.00",549.06,"17,271,431.00"


In [5]:
df.info()  # tipo de cada columna y cuántos valores no vacíos tiene

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18315 entries, 0 to 18314
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   country                   18315 non-null  object 
 1   country_code              18315 non-null  object 
 2   year                      18315 non-null  int64  
 3   sex                       18315 non-null  object 
 4   age_bracket               18315 non-null  object 
 5   suicide_rate_per_100k     18315 non-null  float64
 6   generation                18315 non-null  object 
 7   gdp_usd                   18000 non-null  float64
 8   gdp_per_capita_usd        18000 non-null  float64
 9   total_country_population  18315 non-null  float64
dtypes: float64(4), int64(1), object(5)
memory usage: 1.4+ MB


**¿Qué vemos?**
- El dataset tiene **18,315 filas y 10 columnas**.
- Cada fila dice cuántos suicidios hubo **por cada 100,000 habitantes** en un país, en un año, para un sexo y un grupo de edad.
- También trae el PIB (lo que produce el país en dinero), el PIB por persona y la población del país.
- En `df.info()` solo las dos columnas del PIB tienen menos valores que el total, o sea, tienen datos vacíos.

## 4. Tipos de variables

Hacemos una tabla (diccionario de datos) con cada columna, su tipo y qué significa.

In [6]:
descripcion = {
    "country": "Nombre del país",
    "country_code": "Código de 3 letras del país (ISO)",
    "year": "Año",
    "sex": "Sexo: both (ambos), female (mujeres), male (hombres)",
    "age_bracket": "Grupo de edad (all_ages = todas las edades)",
    "suicide_rate_per_100k": "Suicidios por cada 100,000 habitantes",
    "generation": "Generación",
    "gdp_usd": "PIB del país en dólares",
    "gdp_per_capita_usd": "PIB por persona en dólares",
    "total_country_population": "Población total del país",
}
# Tipo de dato según su significado (no según cómo lo lee pandas)
tipo_conceptual = {
    "country": "Categórica nominal", "country_code": "Categórica nominal (identificador)",
    "year": "Numérica discreta (tiempo)", "sex": "Categórica nominal", "age_bracket": "Categórica ordinal",
    "suicide_rate_per_100k": "Numérica continua", "generation": "Categórica nominal",
    "gdp_usd": "Numérica continua", "gdp_per_capita_usd": "Numérica continua",
    "total_country_population": "Numérica continua",
}
diccionario = pd.DataFrame({
    "variable": df.columns,
    "tipo_pandas": df.dtypes.astype(str).values,
    "tipo_conceptual": [tipo_conceptual[c] for c in df.columns],
    "valores_unicos": df.nunique().values,
    "descripcion": [descripcion[c] for c in df.columns],
})
diccionario

,variable,tipo_pandas,tipo_conceptual,valores_unicos,descripcion
0,country,object,Categórica nominal,185,Nombre del país
1,country_code,object,Categórica nominal (identificador),185,Código de 3 letras del país (ISO)
2,year,int64,Numérica discreta (tiempo),22,Año
3,sex,object,Categórica nominal,3,"Sexo: both (ambos), female (mujeres), male (ho..."
4,age_bracket,object,Categórica ordinal,12,Grupo de edad (all_ages = todas las edades)
5,suicide_rate_per_100k,float64,Numérica continua,3582,"Suicidios por cada 100,000 habitantes"
6,generation,object,Categórica nominal,1,Generación
7,gdp_usd,float64,Numérica continua,4020,PIB del país en dólares
8,gdp_per_capita_usd,float64,Numérica continua,4020,PIB por persona en dólares
9,total_country_population,float64,Numérica continua,4069,Población total del país


In [7]:
# Valores posibles de las columnas de texto más importantes
for col in ["sex", "age_bracket", "generation"]:
    print(f"{col}: {df[col].value_counts().to_dict()}\n")

sex: {'both': 6105, 'female': 6105, 'male': 6105}

age_bracket: {'all_ages': 12210, '10-19': 555, '15-19': 555, '15-29': 555, '20-29': 555, '30-39': 555, '30-49': 555, '40-49': 555, '50-59': 555, '50-69': 555, '60-69': 555, '70plus': 555}

generation: {'All Cohorts': 18315}



**¿Qué vemos?**
- Pandas lee **5 columnas como texto (`object`)**, **1 como entero (`year`)** y **4 como decimales**. Los tipos están bien.
- La población aparece con decimales, pero son personas. La pasamos a entero más adelante.
- **`generation` siempre dice "All Cohorts"**: tiene el mismo valor en todas las filas, así que no nos dice nada.
- En `age_bracket` hay grupos de edad que **se cruzan entre sí** (por ejemplo, `10-19`, `15-19` y `15-29`). Además, `all_ages` aparece muchas más veces que los demás. Lo revisamos en la siguiente sección.

## 5. ¿Qué representa cada fila?

In [8]:
print(f"Países: {df['country'].nunique()} | Códigos: {df['country_code'].nunique()}")
print(f"Años: {df['year'].min()} - {df['year'].max()} ({df['year'].nunique()} años)")
print("Años por país (todos iguales):", df.groupby("country")["year"].nunique().unique())
print()
print("Años disponibles según el tipo de grupo de edad:")
df.assign(tipo=np.where(df["age_bracket"] == "all_ages", "all_ages", "grupo de edad")) \
  .groupby("tipo")["year"].agg(["min", "max", "nunique", "count"])

Países: 185 | Códigos: 185
Años: 2000 - 2021 (22 años)
Años por país (todos iguales): [22]

Años disponibles según el tipo de grupo de edad:


,min,max,nunique,count
tipo,,,,
all_ages,2000,2021,22,12210
grupo de edad,2021,2021,1,6105


**¿Qué vemos?** El dataset tiene **dos tipos de datos mezclados**:

| Tipo de dato | Cómo filtrarlo | Años | Filas |
|---|---|---|---|
| Total del país por año | `age_bracket == "all_ages"` | 2000 a 2021 (22 años) | 12,210 (185 países x 22 años x 3 opciones de sexo) |
| Detalle por grupo de edad | `age_bracket != "all_ages"` | Solo 2021 | 6,105 (185 países x 11 grupos x 3 opciones de sexo) |

Además, `sex = "both"` es el total de hombres y mujeres juntos, no un tercer grupo.

**¿Por qué importa?** Si usamos todas las filas juntas para sacar promedios o para K-Means, **contamos a las mismas personas varias veces**. En la limpieza dejamos todas las filas, pero en las conclusiones explicamos qué filtro usar en cada caso.

## 6. Valores nulos

Revisamos cuántos datos vacíos hay en cada columna.

In [9]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),                          # cantidad de vacíos
    "porcentaje": (df.isna().mean() * 100).round(2),   # % de vacíos
})
print(f"Total de celdas vacías: {df.isna().sum().sum():,} de {df.size:,} ({df.isna().sum().sum() / df.size * 100:.2f}%)")
nulos[nulos["nulos"] > 0]

Total de celdas vacías: 630 de 183,150 (0.34%)


,nulos,porcentaje
gdp_usd,315,1.72
gdp_per_capita_usd,315,1.72


In [10]:
# ¿Faltan siempre juntos el PIB y el PIB por persona?
print("Filas donde falta solo uno de los dos:", (df["gdp_usd"].isna() != df["gdp_per_capita_usd"].isna()).sum())

# El PIB es un dato del país en cada año, así que dejamos una sola fila por país y año
pais_anio = df.drop_duplicates(["country", "year"])[["country", "year", "gdp_usd"]]
faltantes = (pais_anio.assign(falta=pais_anio["gdp_usd"].isna())
             .groupby("country")
             .agg(anios_sin_pib=("falta", "sum"), pct_anios_sin_pib=("falta", "mean"),
                  anios_faltantes=("year", lambda y: list(y[pais_anio.loc[y.index, "gdp_usd"].isna()]))))
faltantes = faltantes[faltantes["anios_sin_pib"] > 0].sort_values("pct_anios_sin_pib", ascending=False)
faltantes["pct_anios_sin_pib"] = (faltantes["pct_anios_sin_pib"] * 100).round(1)
faltantes

Filas donde falta solo uno de los dos: 0


,anios_sin_pib,pct_anios_sin_pib,anios_faltantes
country,,,
"Korea, Democratic People's Republic of",22,100.00,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
South Sudan,14,63.60,"[2000, 2001, 2002, 2003, 2004, 2005, 2006, 200..."
Eritrea,10,45.50,"[2012, 2013, 2014, 2015, 2016, 2017, 2018, 201..."
Yemen,3,13.60,"[2019, 2020, 2021]"
Cuba,1,4.50,[2021]


**¿Qué vemos?**
- Solo faltan datos del PIB: **315 vacíos en cada una de las dos columnas (1.72%)**. Siempre faltan las dos juntas, en las mismas filas.
- Los vacíos son de **solo 5 países**, a los que el Banco Mundial no les tiene esos datos:
  - **Corea del Norte** no tiene PIB en ningún año.
  - **Sudán del Sur** no tiene datos antes de 2008 (se independizó en 2011) ni después de 2015.
  - **Eritrea** no tiene datos desde 2012.
  - **Yemen** no tiene datos de 2019 a 2021.
  - **Cuba** no tiene el dato de 2021.
- Los años que faltan siempre están **al principio o al final**, nunca en medio. Por eso no podemos calcular un valor intermedio; solo podemos repetir el dato del año más cercano.

## 7. Registros duplicados

In [11]:
# Una fila debería ser única por país + año + sexo + grupo de edad
clave = ["country", "year", "sex", "age_bracket"]
print(f"Filas completamente duplicadas: {df.duplicated().sum()}")
print(f"Duplicados por {clave}: {df.duplicated(clave).sum()}")

Filas completamente duplicadas: 0
Duplicados por ['country', 'year', 'sex', 'age_bracket']: 0


**¿Qué vemos?** No hay filas repetidas. Cada combinación de país, año, sexo y grupo de edad aparece una sola vez.

## 8. Inconsistencias

Buscamos datos que estén mal escritos o que no tengan sentido.

### 8.1 Columnas de texto

In [12]:
for col in ["country", "country_code", "sex", "age_bracket", "generation"]:
    s = df[col]
    print(f"{col:<14} espacios_extra={(s != s.str.strip()).sum():<3} "                   # espacios de más
          f"variantes_mayúsc={s.str.lower().nunique() != s.nunique()}")             # p. ej. "Male" vs "male"

print()
print("Todos los códigos tienen 3 letras mayúsculas:", df["country_code"].str.fullmatch(r"[A-Z]{3}").all())
print("Países con más de un código:", (df.groupby("country")["country_code"].nunique() > 1).sum())
print("Códigos con más de un país:", (df.groupby("country_code")["country"].nunique() > 1).sum())
print("Nombres con tildes u otros caracteres especiales:", [c for c in df["country"].unique() if not c.isascii()])

country        espacios_extra=0   variantes_mayúsc=False
country_code   espacios_extra=0   variantes_mayúsc=False
sex            espacios_extra=0   variantes_mayúsc=False
age_bracket    espacios_extra=0   variantes_mayúsc=False
generation     espacios_extra=0   variantes_mayúsc=False

Todos los códigos tienen 3 letras mayúsculas: True
Países con más de un código: 0
Códigos con más de un país: 0
Nombres con tildes u otros caracteres especiales: ["Côte d'Ivoire", 'Türkiye']


**¿Qué vemos?** Los textos están bien. No hay espacios de más ni palabras escritas de dos formas, y cada país tiene un solo código. Los nombres con tildes se leen bien. Algunos nombres están en formato de la ONU (por ejemplo, *"Korea, Republic of"* para Corea del Sur), pero eso es normal.

### 8.2 Valores fuera de rango

Revisamos que cada número esté dentro de lo lógico. Por ejemplo, no puede haber tasas ni poblaciones negativas.

In [13]:
# Rango lógico (mínimo, máximo) de cada columna numérica
reglas = {
    "year": (1950, 2025), "suicide_rate_per_100k": (0, 1000), "gdp_usd": (0, None),
    "gdp_per_capita_usd": (0, None), "total_country_population": (1, None),
}
resultado = []
for col, (lo, hi) in reglas.items():
    s = df[col].dropna()
    n = ((s < lo) | ((s > hi) if hi is not None else False)).sum()
    resultado.append({"variable": col, "min": s.min(), "max": s.max(), "fuera_de_rango": n})
print("Tasas iguales a 0:", (df["suicide_rate_per_100k"] == 0).sum())
pd.DataFrame(resultado)

Tasas iguales a 0: 153


,variable,min,max,fuera_de_rango
0,year,"2,000.00","2,021.00",0
1,suicide_rate_per_100k,0.00,160.59,0
2,gdp_usd,"64,935,850.16","23,725,645,000,000.00",0
3,gdp_per_capita_usd,109.59,"134,965.82",0
4,total_country_population,"74,912.00","1,414,203,896.00",0


In [14]:
# ¿En qué países aparecen las tasas iguales a 0?
df[df["suicide_rate_per_100k"] == 0].groupby("country") \
  .agg(registros=("year", "size"), poblacion_media=("total_country_population", "mean")) \
  .sort_values("registros", ascending=False).head(8)

,registros,poblacion_media
country,,
Antigua and Barbuda,59,"88,361.66"
Saint Vincent and the Grenadines,38,"103,281.32"
Grenada,27,"115,673.22"
Brunei Darussalam,12,"447,360.00"
Cyprus,5,"1,317,309.00"
Armenia,4,"2,962,300.00"
Luxembourg,4,"640,064.00"
Malta,2,"518,150.00"


**¿Qué vemos?** No hay números negativos ni fuera de rango. Hay **153 tasas iguales a 0**, pero casi todas son de países muy pequeños (*Antigua y Barbuda*, *San Vicente y las Granadinas*, *Granada*, *Brunéi*). En un país con poca gente es posible que en un año no haya ningún caso en cierto grupo de edad. Por eso las dejamos.

### 8.3 ¿Las columnas relacionadas coinciden entre sí?

In [15]:
# a) La tasa de 'both' (ambos sexos) debería estar entre la de mujeres y la de hombres
pv = df.pivot_table(index=["country", "year", "age_bracket"], columns="sex", values="suicide_rate_per_100k")
fuera = ((pv["both"] < pv[["female", "male"]].min(axis=1) - 0.01) |
         (pv["both"] > pv[["female", "male"]].max(axis=1) + 0.01)).sum()
print(f"a) Casos donde 'both' NO está entre 'female' y 'male': {fuera} de {len(pv):,}")

# b) El PIB y la población son del país, así que deberían ser iguales en todas las filas de un mismo país y año
print("b) Máximo de valores distintos de PIB/población en un mismo país y año:",
      df.groupby(["country", "year"])[["gdp_usd", "total_country_population"]].nunique().max().max())

# c) El PIB debería ser igual al PIB por persona multiplicado por la población
ratio = df["gdp_usd"] / (df["gdp_per_capita_usd"] * df["total_country_population"])
print(f"c) PIB / (PIB por persona x población): mediana={ratio.median():.3f}, "
      f"min={ratio.min():.3f}, max={ratio.max():.3f}")
print(f"   Filas que se alejan más de un 5%: {((ratio - 1).abs() > 0.05).sum():,} "
      f"({((ratio - 1).abs() > 0.05).mean() * 100:.2f}%)")

a) Casos donde 'both' NO está entre 'female' y 'male': 0 de 6,105


b) Máximo de valores distintos de PIB/población en un mismo país y año: 1
c) PIB / (PIB por persona x población): mediana=1.000, min=0.696, max=1.275
   Filas que se alejan más de un 5%: 189 (1.03%)


In [16]:
# Países donde esa cuenta más se aleja de 1
(df.assign(desviacion=(ratio - 1).abs())
   .groupby("country")["desviacion"].max()
   .sort_values(ascending=False).head(5).to_frame("desviacion_maxima"))

,desviacion_maxima
country,
Cyprus,0.30
Sudan,0.28
Ukraine,0.06
"Tanzania, United Republic of",0.03
Russian Federation,0.02


**¿Qué encontramos?**

| Revisión | Resultado | ¿Qué hacemos? |
|---|---|---|
| a) Tasa de ambos sexos | Siempre está entre la de mujeres y la de hombres. | Nada. |
| b) PIB y población por país y año | Siempre son iguales: son datos del país que se repiten en cada fila de sexo y edad. | Nada, pero hay que recordar que **no cambian según el sexo ni la edad**. |
| c) PIB = PIB por persona x población | Casi siempre se cumple (la mediana da 1.00), pero 189 filas (1.03%) se alejan más de un 5%, sobre todo Chipre y Sudán (hasta un 30%). | Dejamos los valores originales. La población viene de otra fuente (OMS) distinta a la del PIB (Banco Mundial), y por eso no cuadra exacto. |
| Grupos de edad que se cruzan y solo existen en 2021 | No es un error de digitación: así viene armado el dataset. | Los dejamos y explicamos en las conclusiones cómo filtrarlos. |
| `generation` siempre igual | No aporta información. | La borramos. |

## 9. Valores atípicos

Un valor atípico es un dato muy alejado del resto. Para encontrarlos usamos el **método IQR**:
1. Calculamos Q1 (el 25% de los datos está por debajo) y Q3 (el 75% está por debajo).
2. Calculamos IQR = Q3 − Q1.
3. Un dato es atípico si es menor que Q1 − 1.5·IQR o mayor que Q3 + 1.5·IQR.

Para no contar varias veces a la misma población, usamos solo las filas del total del país (`all_ages` y `both`). Aquí **solo los contamos**; los gráficos se hacen en la parte 03.

In [17]:
# Función que cuenta los atípicos de una columna con el método IQR
def atipicos_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr     # límites para considerar algo "normal"
    n = ((s < li) | (s > ls)).sum()
    return {"lim_inf": li, "lim_sup": ls, "min": s.min(), "max": s.max(), "atipicos": n, "%": round(n / len(s) * 100, 2)}

base = df[(df["age_bracket"] == "all_ages") & (df["sex"] == "both")]   # una fila por país y año
filas = {"suicide_rate_per_100k (all_ages, both)": atipicos_iqr(base["suicide_rate_per_100k"])}
for col in ["gdp_usd", "gdp_per_capita_usd", "total_country_population"]:
    filas[col + " (país-año)"] = atipicos_iqr(base[col].dropna())
pd.DataFrame(filas).T

,lim_inf,lim_sup,min,max,atipicos,%
"suicide_rate_per_100k (all_ages, both)",-7.95,25.28,0.00,53.06,172.00,4.23
gdp_usd (país-año),"-227,584,863,458.24","396,808,362,219.44","64,935,850.16","23,725,645,000,000.00",548.00,13.63
gdp_per_capita_usd (país-año),"-17,360.88","32,252.03",109.59,"134,965.82",478.00,11.89
total_country_population (país-año),"-33,030,508.62","61,621,860.38","74,912.00","1,414,203,896.00",468.00,11.50


In [18]:
# Países con las tasas más altas (promedio de 2000 a 2021)
base.groupby("country")["suicide_rate_per_100k"].mean().sort_values(ascending=False).head(8).to_frame("tasa_media")

,tasa_media
country,
Lithuania,36.79
Russian Federation,36.57
Belarus,30.03
Guyana,29.23
"Korea, Republic of",27.28
Kazakhstan,27.06
Ukraine,26.55
Suriname,24.12


**¿Qué vemos?**
- **PIB y población:** unos pocos países tienen valores muchísimo más grandes que el resto (Estados Unidos y China en PIB, China e India en población). Son datos reales, no errores.
- **Tasa de suicidio:** también tiene atípicos altos, en países como Lituania, Rusia, Bielorrusia, Guyana y Corea del Sur, que tienen de las tasas más altas del mundo en promedio de 2000 a 2021. Son datos reales e importantes para el análisis.

**Decisión:** **no borramos** los atípicos. Para K-Means (parte 04) recomendamos aplicar **logaritmo** al PIB, al PIB por persona y a la población antes de escalar, para que los países gigantes no dominen los grupos.

## 10. Limpieza de datos

### 10.1 Borrar la columna `generation`

In [19]:
df = df.drop(columns=["generation"])   # siempre tiene el mismo valor, no aporta nada
print("Columnas actuales:", list(df.columns))

Columnas actuales: ['country', 'country_code', 'year', 'sex', 'age_bracket', 'suicide_rate_per_100k', 'gdp_usd', 'gdp_per_capita_usd', 'total_country_population']


### 10.2 ¿Qué hacemos con el PIB vacío?

Como los años vacíos están al principio o al final, la única forma de rellenarlos es repetir el dato del año más cercano del mismo país. Eso tiene sentido si faltan pocos años, pero no si falta casi toda la serie: estaríamos inventando los datos. Por eso decidimos:

| Caso | Países | ¿Qué hacemos? |
|---|---|---|
| Falta el 40% de los años o menos | Cuba (1 año), Yemen (3 años) | **Rellenar** con el dato del año más cercano del mismo país. |
| Falta más del 40% de los años | Corea del Norte (100%), Sudán del Sur (64%), Eritrea (45%) | **Borrar el país**. Rellenar 10 años o más sería inventar la mayoría de sus datos. |

In [20]:
UMBRAL = 0.40   # 40%

# Porcentaje de años sin PIB de cada país
pct_faltante = (df.drop_duplicates(["country", "year"])
                  .groupby("country")["gdp_usd"].apply(lambda s: s.isna().mean()))
paises_eliminar = pct_faltante[pct_faltante > UMBRAL].index.tolist()
paises_imputar = pct_faltante[(pct_faltante > 0) & (pct_faltante <= UMBRAL)].index.tolist()
print("Países a borrar:   ", paises_eliminar)
print("Países a rellenar: ", paises_imputar)

antes = len(df)
df = df[~df["country"].isin(paises_eliminar)].copy()   # nos quedamos con los demás países
print(f"\nFilas borradas: {antes - len(df)} ({(antes - len(df)) / antes * 100:.2f}%) | quedan: {len(df):,}")

Países a borrar:    ['Eritrea', "Korea, Democratic People's Republic of", 'South Sudan']
Países a rellenar:  ['Cuba', 'Yemen']

Filas borradas: 297 (1.62%) | quedan: 18,018


In [21]:
cols_pib = ["gdp_usd", "gdp_per_capita_usd"]
mask_imputados = df[cols_pib].isna().any(axis=1)   # guardamos cuáles filas vamos a rellenar

# Ordenamos por año y, dentro de cada país, copiamos el dato del año anterior (ffill)
# o, si no hay, el del año siguiente (bfill)
df = df.sort_values(["country", "year"])
df[cols_pib] = df.groupby("country")[cols_pib].transform(lambda s: s.ffill().bfill())

print("Filas rellenadas:", mask_imputados.sum())
df.loc[mask_imputados].drop_duplicates(["country", "year"])[["country", "year"] + cols_pib]

Filas rellenadas: 78


,country,year,gdp_usd,gdp_per_capita_usd
4122,Cuba,2021,"107,352,000,000.00","9,605.28"
18075,Yemen,2019,"21,606,160,663.03",633.89
18078,Yemen,2020,"21,606,160,663.03",633.89
18081,Yemen,2021,"21,606,160,663.03",633.89


### 10.3 Corregir tipos de datos

In [22]:
# La población son personas, así que la pasamos a entero
df["total_country_population"] = df["total_country_population"].round().astype("int64")

# Grupos de edad en orden (de menor a mayor edad)
orden_edad = ["10-19", "15-19", "15-29", "20-29", "30-39", "30-49", "40-49",
              "50-59", "50-69", "60-69", "70plus", "all_ages"]
assert set(df["age_bracket"].unique()) <= set(orden_edad)   # verificamos que no falte ninguno
df["age_bracket"] = pd.Categorical(df["age_bracket"], categories=orden_edad, ordered=True)
df["sex"] = pd.Categorical(df["sex"], categories=["both", "female", "male"])
df["country"] = df["country"].astype("category")
df["country_code"] = df["country_code"].astype("category")

# Ordenamos las filas por país, año, edad y sexo
df = df.sort_values(["country_code", "year", "age_bracket", "sex"]).reset_index(drop=True)
df.dtypes.to_frame("tipo_final")

,tipo_final
country,category
country_code,category
year,int64
sex,category
age_bracket,category
suicide_rate_per_100k,float64
gdp_usd,float64
gdp_per_capita_usd,float64
total_country_population,int64


**Nota:** al guardar en CSV se pierde el tipo `category`. Si alguien necesita el orden de los grupos de edad en otra parte, puede copiar la lista `orden_edad` de este notebook.

## 11. Revisión final y exportación

In [23]:
print(f"Dimensiones finales: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Países: {df['country'].nunique()} | Años: {df['year'].min()}-{df['year'].max()}")
print(f"Vacíos restantes:    {df.isna().sum().sum()}")
print(f"Duplicados:          {df.duplicated(['country', 'year', 'sex', 'age_bracket']).sum()}")

# Si algo de esto falla, el notebook muestra un error
assert df.isna().sum().sum() == 0
assert df.duplicated(["country", "year", "sex", "age_bracket"]).sum() == 0
df.describe().T

Dimensiones finales: 18,018 filas x 9 columnas
Países: 182 | Años: 2000-2021
Vacíos restantes:    0
Duplicados:          0


,count,mean,std,min,25%,50%,75%,max
year,"18,018.00","2,014.00",7.16,"2,000.00","2,008.00","2,016.00","2,021.00","2,021.00"
suicide_rate_per_100k,"18,018.00",10.37,11.45,0.00,3.41,6.79,13.08,160.59
gdp_usd,"18,018.00","415,448,060,416.93","1,801,879,105,022.98","64,935,850.16","8,619,178,773.51","30,469,018,756.57","192,650,021,648.58","23,725,645,000,000.00"
gdp_per_capita_usd,"18,018.00","13,084.18","19,649.30",109.59,"1,463.90","4,541.42","15,509.81","134,965.82"
total_country_population,"18,018.00","39,922,067.36","144,346,785.19","74,912.00","2,489,762.00","8,898,157.50","28,516,770.00","1,414,203,896.00"


In [24]:
# Comparación entre el dataset original y el limpio
resumen = pd.DataFrame({
    "Métrica": ["Filas", "Columnas", "Países", "Celdas vacías", "Filas duplicadas"],
    "Original": [df_raw.shape[0], df_raw.shape[1], df_raw["country"].nunique(),
                 df_raw.isna().sum().sum(), df_raw.duplicated().sum()],
    "Limpio": [df.shape[0], df.shape[1], df["country"].nunique(), df.isna().sum().sum(), df.duplicated().sum()],
})
resumen

,Métrica,Original,Limpio
0,Filas,18315,18018
1,Columnas,10,9
2,Países,185,182
3,Celdas vacías,630,0
4,Filas duplicadas,0,0


In [25]:
# Guardamos el dataset limpio
RUTA_PROCESSED.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RUTA_PROCESSED, index=False, encoding="utf-8")
print(f"Dataset limpio guardado en: {RUTA_PROCESSED} ({len(df):,} registros)")

# Lo volvemos a leer para comprobar que se guardó bien
pd.read_csv(RUTA_PROCESSED).shape

Dataset limpio guardado en: ..\..\data\processed\suicidio_limpio.csv (18,018 registros)


(18018, 9)

## 12. Conclusiones

1. **Cómo es el dataset:** tiene **18,315 filas y 10 columnas**, con **185 países** entre **2000 y 2021**. Cada fila da la tasa de suicidio (por cada 100,000 habitantes) de un país, en un año, para un sexo y un grupo de edad. También trae el PIB, el PIB por persona y la población del país.
2. **Calidad de los datos:** vienen bastante bien. No hay filas repetidas, ni textos mal escritos, ni números imposibles. La tasa de ambos sexos siempre queda entre la de hombres y la de mujeres. Las 153 tasas en 0 son de países pequeños y son válidas.
3. **Datos vacíos:** solo faltaba el PIB (**315 vacíos por columna, 1.72%**), en 5 países:
   - **Cuba y Yemen:** rellenamos los años faltantes con el dato del año más cercano.
   - **Corea del Norte, Sudán del Sur y Eritrea:** los borramos porque les faltaba más del 40% de los años. Son 297 filas, el 1.6% del total.

   Al final no queda ningún vacío.
4. **Columnas:** borramos `generation` porque siempre tenía el mismo valor. Pasamos la población a entero, y el sexo y los grupos de edad a categorías ordenadas.
5. **Cosas raras que anotamos pero no cambiamos:**
   - El PIB no siempre da exacto al multiplicar PIB por persona x población, porque los datos vienen de fuentes distintas.
   - El dataset mezcla datos del total del país con datos por grupo de edad.
6. **Recomendación importante para los compañeros:** no usar todas las filas a la vez, porque se cuenta a la misma gente varias veces. Sugerimos:
   - **Para ver la evolución en el tiempo o comparar países (2000-2021):** filtrar `age_bracket == "all_ages"` y `sex == "both"` (o `female`/`male` para comparar entre sexos).
   - **Para comparar por edad (solo 2021):** usar `age_bracket != "all_ages"` con grupos que no se cruzan: `10-19`, `20-29`, `30-39`, `40-49`, `50-59`, `60-69` y `70plus`.
   - **Para K-Means:** dejar una fila por país (por ejemplo, el promedio de 2000 a 2021 o solo 2021), con la tasa total, la tasa por sexo y los datos económicos. Al PIB y a la población conviene aplicarles **logaritmo**, porque unos pocos países tienen valores enormes.

**Archivo final:** `data/processed/suicidio_limpio.csv` — 18,018 filas x 9 columnas, 182 países, sin vacíos ni duplicados.